%md
![Description](/Workspace/Users/biju.thottathil@3cloudsolutions.com/training/databricksinternaldemo/mcpserver-energy/mcp-claude.png)



Setup Vector Search Index and MCP Server for gold_daily_customer_kwh_summary table
Run this in a Databricks notebook


In [0]:
%sql
-- Table with energy consumption data
select * from `na-dbxtraining`.biju_gold.gold_daily_customer_kwh_summary 

customer_id,first_name,last_name,city,state,plan_id,plan_name,rate_per_kwh,reading_date,total_kwh_daily,num_readings_daily,avg_kwh_per_reading_daily,calculated_cost_daily
CUST00001,Carla,Stewart,Cantuberg,MP,PLAN001,Fixed Rate 12,0.1418,2026-03-02,1.01,1,1.01,0.14
CUST00033,Tabitha,Wallace,Curtiston,UT,PLAN003,Green Energy 24,0.2267,2026-03-02,5.03,1,5.03,1.14
CUST00040,Daniel,Jones,Davidside,RI,PLAN005,Weekend Special,0.2211,2026-03-02,2.46,1,2.46,0.54
CUST00002,Michael,Mullen,Riveraville,MA,PLAN002,Variable Rate 6,0.1603,2026-03-02,2.62,1,2.62,0.42
CUST00003,Christina,Brooks,Millershire,MS,PLAN003,Green Energy 24,0.2267,2026-03-02,3.53,1,3.53,0.8
CUST00004,Jesse,Perez,Suarezbury,AR,PLAN004,Time-of-Use,0.2032,2026-03-02,2.36,1,2.36,0.48
CUST00006,Diamond,Bennett,Donaldshire,NH,PLAN001,Fixed Rate 12,0.1418,2026-03-02,5.97,2,2.98,0.85
CUST00008,Michelle,White,South Tina,KS,PLAN003,Green Energy 24,0.2267,2026-03-02,3.6,1,3.6,0.82
CUST00010,April,Burton,Port Matthew,ME,PLAN005,Weekend Special,0.2211,2026-03-02,1.8,1,1.8,0.4
CUST00012,Jennifer,Cameron,West Annette,VI,PLAN002,Variable Rate 6,0.1603,2026-03-02,6.36,3,2.12,1.02


In [0]:
%sql
CREATE TABLE `na-dbxtraining`.biju_gold.gold_daily_customer_kwh_summary_ne
AS SELECT * FROM `na-dbxtraining`.biju_gold.gold_daily_customer_kwh_summary;

In [0]:

# MAGIC **Added new column to the table called search text for vector search**
# MAGIC 
spark.sql("""
UPDATE `na-dbxtraining`.biju_gold.gold_daily_customer_kwh_summary_ne
SET search_text = CONCAT(
    'Customer: ', COALESCE(first_name, ''), ' ', COALESCE(last_name, ''), '. ',
    'Location: ', COALESCE(city, ''), ', ', COALESCE(state, ''), '. ',
    'Energy plan: ', COALESCE(plan_name, ''), ' (Plan ID: ', COALESCE(plan_id, ''), '). ',
    CASE 
        WHEN total_kwh_daily > 20 THEN 'High energy usage: '
        WHEN total_kwh_daily > 10 THEN 'Medium energy usage: '
        ELSE 'Low energy usage: '
    END,
    CAST(total_kwh_daily AS STRING), ' kWh per day. ',
    'Daily cost: $', CAST(calculated_cost_daily AS STRING), 
    ' at rate $', CAST(rate_per_kwh AS STRING), ' per kWh. ',
    'Reading date: ', CAST(reading_date AS STRING), '.'
)
""")


DataFrame[num_affected_rows: bigint]

In [0]:

#  Enabled Change Data Feed (if not already enabled)
print("\n" + "=" * 80)
print("Step 6: Enable Change Data Feed (if needed)")
print("=" * 80)

# Enable Change Data Feed for automatic index synchronization
spark.sql("""
ALTER TABLE `na-dbxtraining`.biju_gold.gold_daily_customer_kwh_summary_ne
SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")


Step 6: Enable Change Data Feed (if needed)


DataFrame[]

In [0]:

%pip install databricks-vectorsearch
dbutils.library.restartPython()



  Using cached databricks_vectorsearch-0.66-py3-none-any.whl.metadata (2.8 kB)
  Using cached deprecation-2.1.0-py2.py3-none-any.whl.metadata (4.6 kB)
Using cached databricks_vectorsearch-0.66-py3-none-any.whl (21 kB)
Using cached deprecation-2.1.0-py2.py3-none-any.whl (11 kB)
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
#  
# ============================================================================
#Create Vector Search Endpoint
# ============================================================================
print("\n" + "=" * 80)
print("Step 3: Create Vector Search Endpoint")
print("=" * 80)


from databricks.vector_search.client import VectorSearchClient

# Initialize client
vsc = VectorSearchClient(disable_notice=True)

# Endpoint name for gold table
endpoint_name = "customer_kwh_endpoint"

try:
    vsc.create_endpoint(
        name=endpoint_name,
        endpoint_type="STANDARD"
    )
    print(f"✅ Endpoint '{endpoint_name}' created successfully")
except Exception as e:
    if "already exists" in str(e).lower() or "RESOURCE_ALREADY_EXISTS" in str(e):
        print(f"ℹ️  Endpoint '{endpoint_name}' already exists")
    else:
        print(f"⚠️  Error: {e}")



Step 3: Create Vector Search Endpoint
✅ Endpoint 'customer_kwh_endpoint' created successfully


In [0]:
# ============================================================================
#CREATE VECTOR SEARCH INDEX WITH CORRECT PRIMARY KEY
# ============================================================================
print("\n" + "="*80)
print("PART 3: Recreating Vector Search Index")
print("="*80)

from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)

# Configuration
endpoint_name = "customer_kwh_endpoint"
index_name = "na-dbxtraining.biju_gold.customer_kwh_embeddingsindex"
source_table = "na-dbxtraining.biju_gold.gold_daily_customer_kwh_summary_ne"

# Delete existing index if it exists
print("\n🗑️ Deleting existing index (if exists)...")
try:
    vsc.delete_index(endpoint_name=endpoint_name, index_name=index_name)
    print("✅ Existing index deleted successfully")
except Exception as e:
    print(f"ℹ️ Note: {e}")

# Wait a bit for deletion to complete
import time
print("⏳ Waiting for deletion to complete...")
time.sleep(10)

# Create new index with corrected primary key
print(f"\n🔧 Creating new index with primary_key='record_id'...")
try:
    vsc.create_delta_sync_index(
        pipeline_type="TRIGGERED",
        endpoint_name=endpoint_name,
        index_name=index_name,
        primary_key="record_id", 
        source_table_name=source_table,
        embedding_source_column="search_text",
        embedding_model_endpoint_name="databricks-bge-large-en"
    )
    print(f"✅ Vector Search Index '{index_name}' created successfully!")
    print(f"   Primary Key: record_id (customer_id + date)")
    print(f"   Embedding Column: search_text")
    print(f"   Model: databricks-bge-large-en")
except Exception as e:
    print(f"❌ Error creating index: {e}")
    raise


PART 3: Recreating Vector Search Index

🗑️ Deleting existing index (if exists)...
ℹ️ Note: Response content b'{"error_code":"RESOURCE_DOES_NOT_EXIST","message":"Unity Catalog entity na-dbxtraining.biju_gold.customer_kwh_embeddingsindex does not exist.","details":[{"@type":"type.googleapis.com/google.rpc.RequestInfo","request_id":"c6ca21ff-76ef-9e4b-b1f8-f3686c5f4e47","serving_data":""}]}', status_code 404
⏳ Waiting for deletion to complete...

🔧 Creating new index with primary_key='record_id'...
✅ Vector Search Index 'na-dbxtraining.biju_gold.customer_kwh_embeddingsindex' created successfully!
   Primary Key: record_id (customer_id + date)
   Embedding Column: search_text
   Model: databricks-bge-large-en


In [0]:

# ============================================================================
#Test the Vector Search Index
# ============================================================================

print("\n" + "=" * 80)
print("Step 7: Test Vector Search Index")
print("=" * 80)


from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)

endpoint_name = "customer_kwh_endpoint"
index_name = "na-dbxtraining.biju_gold.customer_kwh_embeddingsindex"

# Get the index
index = vsc.get_index(
    endpoint_name=endpoint_name,
    index_name=index_name
)

# Test search - query for high energy usage customers
results = index.similarity_search(
    query_text="high energy usage customer in California",
    columns=["customer_id", "first_name", "last_name", "city", "state", "plan_name", "total_kwh_daily", "calculated_cost_daily", "reading_date"],
    num_results=5
)

print("🔍 Search Results:")
if 'result' in results and 'data_array' in results['result']:
    for i, result in enumerate(results['result']['data_array'], 1):
        print(f"\\n{i}. Customer: {result[1]} {result[2]}")
        print(f"   Location: {result[3]}, {result[4]}")
        print(f"   Plan: {result[5]}")
        print(f"   Daily Usage: {result[6]:.2f} kWh")
        print(f"   Daily Cost: ${result[7]:.2f}")
        print(f"   Date: {result[8]}")
else:
    print("No results found")




Step 7: Test Vector Search Index
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
🔍 Search Results:
\n1. Customer: Tammy Braun
   Location: South Morgan, CA
   Plan: Time-of-Use
   Daily Usage: 3.96 kWh
   Daily Cost: $0.80
   Date: 2026-03-02
\n2. Customer: Jason Robinson
   Location: North John, HI
   Plan: Weekend Special
   Daily Usage: 0.35 kWh
   Daily Cost: $0.08
   Date: 2026-03-02
\n3. Customer: Christopher Campos
   Location: East Debra, OK
   Plan: Variable Rate 6
   Daily Usage: 1.53 kWh
   Daily Cost: $0.25
   Date: 2026-03-02
\n4. Customer: Rachel Baker
   Location: Danielsstad, VA
   Plan: Green Energy 24
   Daily Usage: 7.72 kWh
   Daily Cost: $1.75
   Date: 2026-03-02
\n5. Customer: Julie White
   Location: South Samuelland, WI
   Plan: Time-of-Use
   Daily Usage: 3.93 kWh
   Daily Cost: $0.80
   Date: 2026-03

In [0]:
# ============================================================================
# MCP SERVER URL GENERATION 
# ============================================================================

import json

# Step 8: MCP Server URL Generation
print("\n" + "=" * 80)
print("Step 8: MCP Server Configuration")
print("=" * 80)

# Configuration
DATABRICKS_HOST = "https://adb-1952652121322753.13.azuredatabricks.net"
CATALOG = "na-dbxtraining"
SCHEMA = "biju_gold"
INDEX_NAME = "customer_kwh_embeddingsindex"
MCP_SERVER_NAME = "biju_gold"

# Generate MCP Server URL
MCP_SERVER_URL = f"{DATABRICKS_HOST}/api/2.0/mcp/vector-search/{CATALOG}/{MCP_SERVER_NAME}"
MCP_TOOL_NAME = f"{CATALOG}__{SCHEMA}__{INDEX_NAME}"

print(f"\n✅ MCP Server URL:\n   {MCP_SERVER_URL}")
print(f"\n🔧 MCP Tool Name:\n   {MCP_TOOL_NAME}")

# Claude Desktop Configuration
claude_config = {
    "mcpServers": {
        MCP_SERVER_NAME: {
            "type": "url",
            "url": MCP_SERVER_URL,
            "name": MCP_SERVER_NAME
        }
    }
}

print(f"\n📝 Add to claude_desktop_config.json:")
print(json.dumps(claude_config, indent=2))

print("\n" + "=" * 80)
print("MCP Server Ready!")
print("=" * 80)


Step 8: MCP Server Configuration

✅ MCP Server URL:
   https://adb-1952652121322753.13.azuredatabricks.net/api/2.0/mcp/vector-search/na-dbxtraining/biju_gold

🔧 MCP Tool Name:
   na-dbxtraining__biju_gold__customer_kwh_embeddingsindex

📝 Add to claude_desktop_config.json:
{
  "mcpServers": {
    "biju_gold": {
      "type": "url",
      "url": "https://adb-1952652121322753.13.azuredatabricks.net/api/2.0/mcp/vector-search/na-dbxtraining/biju_gold",
      "name": "biju_gold"
    }
  }
}

MCP Server Ready!


In [0]:
%sql
SELECT customer_id, first_name, last_name, city, state, total_kwh_daily, calculated_cost_daily, plan_name
FROM `na-dbxtraining`.biju_gold.gold_daily_customer_kwh_summary_ne
WHERE state = 'AR'
ORDER BY total_kwh_daily DESC
LIMIT 10

customer_id,first_name,last_name,city,state,total_kwh_daily,calculated_cost_daily,plan_name
CUST00037,Tiffany,Woodard,Stephenchester,AR,8.28,1.33,Variable Rate 6
CUST00004,Jesse,Perez,Suarezbury,AR,2.36,0.48,Time-of-Use
